# Closed-form reference for a 2-D thermal plant

Analytical Green's-function solutions for the same diffusion-decay PDE that the
`disk*.py` controllers learn from hardware. Use this notebook as the ground-truth
check for the learned surrogates.

- `temperature` — transient solution with point thermostats and Newton cooling.
- `temperature_steady` — steady state with finite thermostat coupling.
- `temperature_steady_pt` — differentiable PyTorch version of the steady state.

No lab hardware or capture files are needed. The notebook ships without stored
outputs; work through it from the top. Some cells are deliberately expensive.

In [ ]:
import numpy as np
from scipy.integrate import quad
def temperature(x, y, t, sources=((0.,0.,100.,1.),), kappa=1e-4, h_env=0, eps=1e-45):
    """
    Compute T(x,y,t) on an infinite 2D plane initially at 0,
    with independent point-thermostats and environmental cooling.
    Parameters
    ----------
    x, y : float
        Query coordinates.
    t : float
        Time > 0.
    sources : list of tuples
        Each tuple is (x_i, y_i, T_i, lambda_i):
          - (x_i,y_i): location
          - T_i: set-point temperature at that thermostat
          - lambda_i: coupling strength (s^-1), independent of h_env
    kappa : float
        Thermal diffusivity (m^2/s).
    h_env : float
        Environmental Newton-cooling rate (s^-1).
    eps : float
        Gaussian regularization radius for delta sources (m).
    Returns
    -------
    T_val : float
        Temperature at (x,y,t), guaranteed in [0, max T_i].
    """
    if h_env < 0:
        raise ValueError("h_env must be non-negative")
    def integrand(s, xi, yi, Ti, lam):
        tau = t - s
        # 2D heat kernel denominator regularized by eps:
        denom = 4*kappa*tau + 2*eps*eps
        # distance from (x,y) to (xi,yi)
        r = np.hypot(x-xi, y-yi)
        expo = np.exp(-h_env*tau - r*r/denom)
        # source term = lam * (T_i - local T ≈ T_i initially)
        return lam * Ti * expo / denom
    T_val = 0.0
    for xi, yi, Ti, lam in sources:
        part, _ = quad(
            integrand, 0, t,
            args=(xi, yi, Ti, lam),
            epsabs=0, epsrel=1e-4, limit=200
        )
        T_val += part
    # numerical clamp
    return min(max(T_val, 0.0), max(src[2] for src in sources))
o = np.array([[[temperature(x,y,t)if x<=y else np.nan for y in np.linspace(0,1,11)[1:]]for x in np.linspace(0,1,11)[1:]]for t in np.linspace(10,500,20)], 'f')

In [ ]:
temperature(100,100,9e9,h_env=1e-6)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(o[-1])
plt.colorbar()

In [ ]:
import numpy as np
from scipy.special import k0
def temperature_steady(x, y, sources, kappa=1e-4, h_env=1e-4, eps=1e-6):
    """
    Steady-state T(x,y) with finite-lambda thermostats and Newton cooling.
    Parameters
    ----------
    x, y : scalar or array_like
        Query coordinates; arrays must broadcast to same shape.
    sources : list of tuples
        Each tuple is (xi, yi, Ti, lambda_i).
    kappa : float
        Thermal diffusivity (>0).
    h_env : float
        Ambient cooling rate (>=0).
    eps : float
        Small regularization for r=0 in kernel (default 1e-6).
    Returns
    -------
    T_inf : float or ndarray
        Steady-state temperature(s) at (x,y), clipped to [0, max(Ti)].
    """
    # unpack
    xs = np.array([s[0] for s in sources], float)
    ys = np.array([s[1] for s in sources], float)
    Ts = np.array([s[2] for s in sources], float)
    lam = np.array([s[3] for s in sources], float)
    N = len(sources)
    # build pairwise distances and kernel H_ij
    # r2_ij = (xi-xj)^2 + (yi-yj)^2, but add eps^2 only on diag
    dx = xs[:,None] - xs[None,:]
    dy = ys[:,None] - ys[None,:]
    r2 = dx*dx + dy*dy
    # regularize only where i==j
    r2 += np.eye(N)*eps*eps
    scale = np.sqrt(h_env/kappa)
    H = (1.0/(2*np.pi*kappa)) * k0(np.sqrt(r2)*scale)
    # finite-lambda matrix L_{ij} = lambda_j * H_ij
    L = H * lam[None,:]
    # solve (I + L) T* = L T  -->  T* = (I+L)^{-1} L T
    A = np.eye(N) + L
    b = L.dot(Ts)
    T_src = np.linalg.solve(A, b)  # T*(xi,yi)
    # now evaluate at query points
    xq, yq = np.broadcast_arrays(x, y)
    T_out = np.zeros_like(xq, float)
    # for each source, compute its contribution
    prefac = 1.0/(2*np.pi*kappa)
    for (xi, yi, Ti, li), Tsi in zip(sources, T_src):
        # weight = lambda_i * (T_i - T_i^*)
        weight = li * (Ti - Tsi)
        # compute distance r from each query point
        dxq = xq - xi
        dyq = yq - yi
        rq = np.hypot(dxq, dyq)
        # wherever rq==0, replace by eps to avoid K0 singularity
        rq = np.where(rq==0, eps, rq)
        T_out += weight * prefac * k0(rq * scale)
    # clamp for numerical safety
    np.clip(T_out, 0.0, Ts.max(), out=T_out)
    return T_out
temperature_steady(.1, .1, [(0,0,300.,1.),(1,1,300.,1.),(2,2,300.,1.)])

In [ ]:
import torch
def temperature_steady_pt(x, y, sources, kappa, h_env, eps=1e-6):
    """
    Steady-state T(x,y) solver in PyTorch, fully differentiable.
    Parameters
    ----------
    x, y : Tensor, shape (...,)
        Query coordinates.  Can be scalar tensors or broadcastable arrays.
    sources : Tensor, shape (N, 4)
        Each row is [xi, yi, Ti, lambda_i].
    kappa : Tensor, scalar or shape (...,)
        Thermal diffusivity (>0).
    h_env : Tensor, scalar or shape (...,)
        Ambient cooling rate (>=0).
    eps : float
        Small regularization on the diagonal of the kernel.
    Returns
    -------
    T_out : Tensor, shape (...)
        Steady-state temperature at each (x,y).
    """
    # unpack sources
    xi = sources[:, 0]
    yi = sources[:, 1]
    Ti = sources[:, 2]
    lam = sources[:, 3]
    N = xi.shape[0]
    # ensure x,y broadcast to same shape
    xq, yq = torch.broadcast_tensors(x, y)
    # pairwise distances squared between sources, regularize diagonal
    dx = xi[:, None] - xi[None, :]
    dy = yi[:, None] - yi[None, :]
    r2 = dx*dx + dy*dy + torch.eye(N, device=dx.device) * (eps*eps)
    # scale factor for the kernel argument
    scale = torch.sqrt(h_env / kappa)
    arg = torch.sqrt(r2) * scale
    # kernel matrix H_ij = (1 / (2π κ)) * K0(arg)
    H = torch.special.modified_bessel_k0(arg) / (2 * torch.pi * kappa)
    # finite-λ coupling matrix L_{ij} = λ_j * H_ij
    L = H * lam[None, :]
    # solve for on-source temperatures T_src:  (I + L) T_src = L @ Ti
    A = torch.eye(N, device=L.device) + L
    b = L.matmul(Ti)
    T_src = torch.linalg.solve(A, b)
    # now compute T_out at query points
    T_out = torch.zeros_like(xq)
    prefac = 1.0 / (2 * torch.pi * kappa)
    for j in range(N):
        weight = lam[j] * (Ti[j] - T_src[j])
        dxq = xq - xi[j]
        dyq = yq - yi[j]
        rq = torch.sqrt(dxq*dxq + dyq*dyq)
        rq = torch.where(rq == 0, eps, rq)
        T_out = T_out + weight * prefac * torch.special.modified_bessel_k0(rq * scale)
    # clamp for numerical safety
    Tmin, Tmax = Ti.min(), Ti.max()
    return torch.clamp(T_out, Tmin, Tmax)
# example parameters
sources = torch.tensor([
    [0.0, 0.0, 100.0, 1.0],    # (xi, yi, Ti, lambda_i)
    [1.0, 0.0,  50.0, 0.5],
], dtype=torch.double, requires_grad=True)
kappa  = torch.tensor(1e-4, dtype=torch.double, requires_grad=True)
h_env   = torch.tensor(1e-3, dtype=torch.double, requires_grad=True)
eps     = 1e-6
# query points
xq = torch.linspace(-2, 3, 100, dtype=torch.double)
yq = torch.zeros_like(xq)
# compute steady-state
T = temperature_steady_pt(xq, yq, sources, kappa, h_env, eps)
# now you can backpropagate: e.g.
loss = (T.mean() - 20.0)**2
loss.backward()
print("dLoss/dkappa =", kappa.grad)